In [ ]:
#!pip install pykeen
##check installation - using Python environment by default as CUDA and Pytorch need to be compatible
#import pykeen

In [4]:
!python --version

Python 3.10.8


## PyKEEN implementations of embedding models and evaluation pipeline with NP-KG v2.1
1. Experiment 1: Generate embeddings from NP-KG and evaluate with existing edges in NP-KG and save embeddings
2. Experiment 2: Include hyperparameter optimization
3. Experiment 3: ...

## Load data

TSV file with head, relation, tail columns

In [5]:
from pykeen.triples import TriplesFactory

/ihome/crc/install/pytorch/2.0.1/python3.10/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
KG_PATH = '/ihome/rboyce/sbt12/np-kg/resources/knowledge_graphs/'
DIR_PATH = 'pykeen-models/'

In [7]:
#Time taken ~ 1 min
tf = TriplesFactory.from_path(KG_PATH+'NP-KG-CURIE-only-v2.0.0.tsv')

In [8]:
tf.num_entities

1090175

In [9]:
tf.num_relations

354

In [10]:
tf.num_triples

7934519

In [11]:
npkg_train, npkg_test, npkg_val = tf.split([.8, .1, .1], random_state=42)

In [12]:
print(npkg_train.num_triples, npkg_test.num_triples, npkg_val.num_triples)

6347615 793452 793452


In [ ]:
##CUDA version for CRC GPU is 11.7 or 11.8
##Pytorch needs to be compatible with CUDA (default is for CUDA 12.1)

In [1]:
import torch

In [2]:
#pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
torch.cuda.is_available()

True

In [3]:
torch.version.cuda

'11.8'

## Load models and pipeline

In [13]:
from pykeen.pipeline import pipeline
from pykeen.models import TransE, TransR, TransD, RotatE, RGCN
#from pykeen.datasets import Nations

In [ ]:
##using existing data
from pykeen.datasets import Nations
pipeline_result = pipeline(
    dataset=Nations,
    model=TransE,
    random_seed=42,
    device='cpu'
)
pipeline_result.save_to_directory('nations_transe')

### TransE

In [ ]:
##Pipeline
# Default TransE parameters: embedding_dim=50, scoring_fct_norm=1, 
# Default pipeline parameters: regularizer=None, lr_scheduler=ExponentialLR, optimizer=Adagrad, training_loop='slcwa', negative_sampler='basic',  
# stopper=None, evaluator='RankBasedEvaluator'
# Time taken with CRC GPU (1 core): 
transe_result = pipeline(
    training=npkg_train,
    testing=npkg_test,
    validation=npkg_val,
    model=TransE,
    model_kwargs=dict(embedding_dim=100),
    epochs=100,
    stopper='early',
    random_seed=42,
)

Training batches on cuda:0: 100%|█████████▉| 24781/24796 [05:35<00:00, 74.11batch/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)


Training batches on cuda:0:  82%|████████▏ | 20428/24796 [04:36<00:58, 74.11batch/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)


Training batches on cuda:0:   8%|▊         | 2025/24796 [00:27<05:07, 74.02batch/s]


In [ ]:
transe_result.save_to_directory(DIR_PATH+'npkg_transe_default_pykeen')

In [ ]:
import torch

In [ ]:
#load model
transe_model = torch.load(MODEL_PATH+'npkg_transe_default_pykeen/npkg_transe_default_pykeen.pkl')

In [ ]:
##get evaluation results

In [ ]:
'''-- not the full list
Parameters and assumptions:
- training_loop='sLCWA' (default - stochastic local closed world assumption) or 'LCWA' (local closed world assumption)
- negative_sampler='basic', 'bernoulli'
- evaluator='RankBasedEvaluator' (default), MacroRankBasedEvaluator, SampledRankBasedEvaluator, ClassificationEvaluator
- stopper='early'
- lr_scheduler='ExponentialLR'
- lr_schedule_kwargs={'gamma': 0.99}
- model_kwargs={
    'scoring_fct_norm=2,
} and other model specific params
- training_kwargs={
    'num_epochs': 
}
- loss='softplus', 
- optimizer_kwargs={
    'lr': 0.01
} 
- evaluation_entity_whitelist=Collection of string of entitites for restricting evaluation to triples only containing these entities
- evaluation_relation_whitelist=Collection of string of relations for restricting evaluation to triples only containing these relations
'''

### TransR

## Hyperparameter optimization

Can be done for model, loss, regularizer, optimizer, negative_sampler and training. Each model has a HPO strategy defined in the package.
* TransE model’s default HPO strategy for its embedding_dim argument is to search between [16, 256] with a step size of 16. The norm is set to search as either 1 or 2.

In [ ]:
from pykeen.hpo import hpo_pipeline

In [ ]:
hpo_pipeline_result = hpo_pipeline(
    n_trials=30,
    dataset='Nations',
    model='TransE',
)

In [ ]:
#See https://pykeen.readthedocs.io/en/stable/tutorial/first_steps.html#mapping-entity-and-relation-identifiers-to-their-names

## Using embeddings

In [ ]:
from typing import List
import pykeen.nn

In [ ]:
embed_model = transe_result.model
entity_representation_modules: List['pykeen.nn.Representation'] = embed_model.entity_representations
relation_representation_modules: List['pykeen.nn.Representation'] = embed_model.relation_representations

In [ ]:
entity_embeddings: pykeen.nn.Embedding = entity_representation_modules[0]
relation_embeddings: pykeen.nn.Embedding = relation_representation_modules[0]

In [ ]:
entity_embedding_tensor: torch.FloatTensor = entity_embeddings()
relation_embedding_tensor: torch.FloatTensor = relation_embeddings()
entity_embedding_tensor = model.entity_representations[0](indices=None).detach().numpy()
relation_embedding_tensor = model.relation_representations[0](indices=None).detach().numpy()